# 🕹️ Retro Arcade — Check & Score

> Build stamp: **2026-05-30 09:49:13**

Reads the **PBIR definition** of your report via `sempy.fabric.get_report_definition`,
grades each of the 5 levels, assigns a rank, and mints your signed badge.

Re-run any time — the report is fetched fresh every run.


## Step 0 — Configure


In [ ]:
# ============================================================
# 👇 Edit these two values before running
# ============================================================
PLAYER_NAME  = "Your Name Here"           # name shown on the badge
REPORT_NAME  = "Arcade_Hall_Report"       # name you gave your report
# WORKSPACE is auto-detected (current workspace)
# ============================================================


## Step 1 — Install / import dependencies


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0"],
               check=False, capture_output=True)
try:
    import sempy.fabric as fabric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link"],
                   check=True)
    import sempy.fabric as fabric
try:
    import sempy_labs as labs
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link-labs"],
                   check=True)
    import sempy_labs as labs
import json, base64
from collections import Counter


## Step 2 — Fetch the report definition (PBIR)


In [ ]:
# Call Fabric REST API directly (avoids sempy/sempy_labs version mismatches).
import time, requests, notebookutils
ws_id = notebookutils.runtime.context["currentWorkspaceId"]
token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
H = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
BASE = "https://api.fabric.microsoft.com/v1"

# Find the report id by name
r = requests.get(f"{BASE}/workspaces/{ws_id}/items?type=Report", headers=H, timeout=60)
r.raise_for_status()
items = r.json().get("value", [])
match = [it for it in items if it.get("displayName") == REPORT_NAME]
if not match:
    raise RuntimeError(f"Report '{REPORT_NAME}' not found in workspace {ws_id}")
report_id = match[0]["id"]
print(f"📥 Fetching '{REPORT_NAME}' (id={report_id})...")

def _call_get_definition(fmt=None):
    url = f"{BASE}/workspaces/{ws_id}/items/{report_id}/getDefinition"
    if fmt:
        url += f"?format={fmt}"
    r = requests.post(url, headers=H, timeout=60)
    if r.status_code == 200:
        return r.json().get("definition", {})
    if r.status_code == 202:
        op_url = r.headers.get("Location") or r.headers.get("location")
        for _ in range(60):
            time.sleep(2)
            pr = requests.get(op_url, headers=H, timeout=60)
            pr.raise_for_status()
            body = pr.json()
            st = body.get("status", "").lower()
            if st in ("succeeded", "completed"):
                result_url = op_url + "/result" if not op_url.endswith("/result") else op_url
                rr = requests.get(result_url, headers=H, timeout=60)
                rr.raise_for_status()
                return rr.json().get("definition", {})
            if st in ("failed", "cancelled"):
                raise RuntimeError(f"getDefinition LRO {st}: {pr.text}")
        raise TimeoutError("getDefinition LRO timed out")
    raise RuntimeError(f"getDefinition HTTP {r.status_code}: {r.text}")

report_format = "PBIR"
try:
    definition = _call_get_definition("PBIR")
except RuntimeError as e:
    if "FailedToExportReport" in str(e) or "cannot be converted" in str(e):
        print("⚠️  Report is in legacy format (not PBIR). Falling back to default export.")
        report_format = "LEGACY"
        definition = _call_get_definition(None)
    else:
        raise

# Normalize into dict {path: text}
parts = {}
for part in definition.get("parts", []):
    p = part["path"]
    payload = part.get("payload", "")
    ptype = (part.get("payloadType") or "").lower()
    try:
        if ptype == "inlinebase64":
            raw = base64.b64decode(payload).decode("utf-8", errors="replace")
        else:
            raw = str(payload)
    except Exception:
        raw = str(payload)
    parts[p] = raw

print(f"✅ Got {len(parts)} parts (format={report_format}).")


## Step 3 — Parse pages, visuals, theme, mobile layout


In [ ]:
# ----------------------------------------------------------------
# Build a normalized view (pages_data, all_visuals, etc.) that works
# for BOTH PBIR (exploded parts) and LEGACY (single report.json with
# serialized layout in 'report.layout').
# ----------------------------------------------------------------
def _json(path):
    if path in parts:
        try: return json.loads(parts[path])
        except Exception: return None
    return None

pages_data = []
report_json = {}
theme_paths = []
has_custom_theme = False
bookmarks_root = None

if report_format == "LEGACY":
    # Legacy: single 'report.json' with a nested 'layout' string (JSON).
    rj = _json("report.json") or _json("definition/report.json") or {}
    layout_raw = rj.get("layout")
    if isinstance(layout_raw, str):
        try: layout = json.loads(layout_raw)
        except Exception: layout = {}
    else:
        layout = layout_raw or rj
    report_json = layout

    for section in (layout.get("sections") or []):
        sec_cfg_raw = section.get("config")
        try:
            sec_cfg = json.loads(sec_cfg_raw) if isinstance(sec_cfg_raw, str) else (sec_cfg_raw or {})
        except Exception:
            sec_cfg = {}
        sec_filters_raw = section.get("filters")
        try:
            sec_filters = json.loads(sec_filters_raw) if isinstance(sec_filters_raw, str) else (sec_filters_raw or [])
        except Exception:
            sec_filters = []

        # Build a single big string with ALL section JSON to do permissive scans.
        sec_blob = json.dumps({"section": section, "cfg": sec_cfg, "filters": sec_filters}, default=str)

        # Tooltip detection (legacy). PBI stores tooltip pages with one of:
        #   - sec_cfg.objects.pageInformation[0].properties.type.expr.Literal.Value == "'Tooltip'"
        #   - section.height==300 width==320 + altTextCollection.type=='Tooltip'
        #   - displayOption == 3
        is_tooltip = (
            "'Tooltip'" in sec_blob
            or '"Tooltip"' in sec_blob
            or section.get("displayOption") == 3
        )

        # Drillthrough detection (legacy):
        #   - filter with type == "Drillthrough" (string OR enum int 5)
        #   - sec_cfg.objects.pageInformation[*].type literal 'Drillthrough'
        #   - heuristic: section.filters is non-trivial (>20 chars JSON) and not a tooltip
        is_drill = (
            "'Drillthrough'" in sec_blob
            or '"Drillthrough"' in sec_blob
            or '"type":"Passthrough"' in sec_blob
        )
        for f in (sec_filters if isinstance(sec_filters, list) else []):
            if isinstance(f, dict):
                t = f.get("type")
                if (isinstance(t, str) and t.lower() == "drillthrough") or t == 5:
                    is_drill = True
        # Heuristic fallback: a regular full-size page with non-trivial filters
        # is almost certainly a drillthrough target (filters at page level are
        # what define drillthrough pages in PBI).
        if not is_drill and not is_tooltip:
            _fl_raw = section.get("filters")
            _flen = len(_fl_raw) if isinstance(_fl_raw, str) else (
                len(json.dumps(_fl_raw)) if _fl_raw else 0
            )
            if _flen > 20:
                is_drill = True

        pjson = {
            "name": section.get("name", ""),
            "displayName": section.get("displayName", ""),
            "objects": (sec_cfg.get("objects") if isinstance(sec_cfg, dict) else {}) or {},
            "_section_raw": section,
            "_legacy_kind": "tooltip" if is_tooltip else ("drillthrough" if is_drill else "regular"),
            "_has_mobile": (section.get("displayOption") == 1) or
                           ('"mobile"' in (sec_cfg_raw if isinstance(sec_cfg_raw, str) else "")) or
                           ("mobileLayout" in sec_blob),
        }
        visuals = []
        for vc in (section.get("visualContainers") or []):
            vconfig = vc.get("config")
            try:
                vconfig = json.loads(vconfig) if isinstance(vconfig, str) else (vconfig or {})
            except Exception:
                vconfig = {}
            visuals.append({
                "visual": {"visualType": (vconfig.get("singleVisual") or {}).get("visualType")},
                "_raw": vc,
                "_config": vconfig,
            })
        pages_data.append({"name": pjson["name"], "json": pjson, "visuals": visuals})

    # theme (legacy)
    if isinstance(rj.get("themeCollection"), dict) or rj.get("theme"):
        has_custom_theme = True

    # bookmarks (legacy): live in report.config (a JSON string) under 'bookmarks'
    rcfg_raw = rj.get("config")
    try:
        rcfg = json.loads(rcfg_raw) if isinstance(rcfg_raw, str) else (rcfg_raw or {})
    except Exception:
        rcfg = {}
    bookmarks_root = rcfg.get("bookmarks") or layout.get("bookmarks") or rj.get("bookmarks")
    # custom theme can also be flagged inside report.config
    if not has_custom_theme:
        if isinstance(rcfg, dict) and (rcfg.get("themeCollection") or rcfg.get("activeSectionIndex") is not None and rcfg.get("theme")):
            if rcfg.get("themeCollection") or rcfg.get("theme"):
                has_custom_theme = True
else:
    # PBIR
    pages_index = _json("definition/pages/pages.json") or {}
    page_names = []
    if isinstance(pages_index.get("pageOrder"), list):
        page_names = list(pages_index["pageOrder"])
    else:
        for p in parts:
            if p.startswith("definition/pages/") and p.endswith("/page.json"):
                page_names.append(p.split("/")[2])
        page_names = sorted(set(page_names))

    for pn in page_names:
        pjson = _json(f"definition/pages/{pn}/page.json") or {}
        pjson["_has_mobile"] = any(
            p.startswith(f"definition/pages/{pn}/mobile") for p in parts
        )
        visuals = []
        for p in parts:
            prefix = f"definition/pages/{pn}/visuals/"
            if p.startswith(prefix) and p.endswith("/visual.json"):
                vj = _json(p) or {}
                visuals.append(vj)
        pages_data.append({"name": pn, "json": pjson, "visuals": visuals})

    report_json = _json("definition/report.json") or {}
    theme_paths = [p for p in parts if p.startswith("StaticResources/RegisteredResources/")
                                       and p.endswith(".json")]
    has_custom_theme = bool(theme_paths)

print(f"📄 Pages found: {len(pages_data)}")
for pd in pages_data:
    kind = pd["json"].get("_legacy_kind", "regular")
    mob  = pd["json"].get("_has_mobile", False)
    print(f"   - {pd['name']!r}  display={pd['json'].get('displayName','')!r}  kind={kind}  mobile={mob}  visuals={len(pd['visuals'])}")


## Step 4 — Grade the 5 levels


In [ ]:
SLICER_KEYS = ("slicer", "advancedSlicerVisual")

def visual_type(v):
    # PBIR shape varies; check common fields
    return (v.get("visual", {}).get("visualType")
            or v.get("visualContainerObjects", {}).get("visualType")
            or v.get("singleVisual", {}).get("visualType")
            or v.get("visualType")
            or "")

def page_display_title(pj):
    # PBIR shape: pj['displayName'] is the page title shown in the tab
    return pj.get("displayName") or ""

def page_has_background(pj):
    # check pj['objects']['background'] presence (color or image)
    objs = pj.get("objects") or {}
    bg   = objs.get("background")
    return bool(bg)

def page_kind(pj):
    # Legacy: use the _legacy_kind we computed during parsing
    if pj.get("_legacy_kind"):
        return pj["_legacy_kind"]
    # Tooltip / Drillthrough markers (PBIR)
    opt = pj.get("type") or pj.get("pageType") or ""
    if str(opt).lower() == "tooltip": return "tooltip"
    if str(opt).lower() == "drillthrough" or pj.get("filterConfig", {}).get("filters"):
        for f in (pj.get("filterConfig", {}).get("filters") or []):
            if str(f.get("type", "")).lower() == "drillthrough":
                return "drillthrough"
    return "regular"

def page_has_mobile(pj):
    # PBIR mobile layout is a separate JSON: definition/pages/<page>/mobile.json
    if pj.get("_has_mobile"):
        return True
    return any(p.startswith(f"definition/pages/{pj.get('name','')}/mobile") for p in parts)

# ---------------- Level 1 ---------------- #
n_pages = len(pages_data)
titled  = sum(1 for pd in pages_data if page_display_title(pd["json"]).strip()
              and not page_display_title(pd["json"]).strip().lower().startswith("page "))
bg      = sum(1 for pd in pages_data if page_has_background(pd["json"]))
L1 = 0
L1 += 10 if n_pages >= 3 else (5 if n_pages == 2 else 0)
L1 += 5  if titled >= max(3, n_pages) else (3 if titled >= 2 else 0)
L1 += 5  if bg >= max(3, n_pages) else (3 if bg >= 1 else 0)
L1 = min(20, L1)
print(f"🟢 L1 Foundation:      pages={n_pages}  titled={titled}  bg={bg}  →  {L1}/20")

# ---------------- Level 2 ---------------- #
all_visuals = [v for pd in pages_data for v in pd["visuals"]]
types = Counter(visual_type(v) for v in all_visuals if visual_type(v))
n_unique = sum(1 for t,c in types.items() if c >= 1)
has_card   = any(t in ("card","cardVisual","multiRowCard") for t in types)
has_slicer = any(t in SLICER_KEYS for t in types)
L2 = min(20, n_unique * 4)
if not has_card:   L2 = min(L2, 16)
if not has_slicer: L2 = min(L2, 16)
print(f"🟡 L2 Visuals:         types_seen={dict(types)}  unique={n_unique}  card={has_card}  slicer={has_slicer}  →  {L2}/20")

# ---------------- Level 3 ---------------- #
n_slicers = sum(c for t,c in types.items() if t in SLICER_KEYS)
# sync slicer: a slicer with syncSlicers configured (very approximate)
sync_count = 0
for v in all_visuals:
    if visual_type(v) in SLICER_KEYS:
        s = json.dumps(v)
        if '"syncGroup"' in s or '"syncSlicers"' in s:
            sync_count += 1
# edit-interactions: in LEGACY they live as section.visualInteractions = [{source,target,typeByVisual}]
# but PBI sometimes serializes them inside section.config.visualInteractions instead.
# In PBIR each visual.json may have 'visualContainerObjects.visualInteractions' / 'interactionType'.
interactions = 0
for pd in pages_data:
    sec = pd["json"].get("_section_raw") or {}
    # 1) Direct on section
    vi = sec.get("visualInteractions")
    if isinstance(vi, list):
        interactions += len(vi)
    # 2) Inside section.config (string-encoded JSON)
    cfg_raw = sec.get("config")
    try:
        cfg = json.loads(cfg_raw) if isinstance(cfg_raw, str) else (cfg_raw or {})
    except Exception:
        cfg = {}
    if isinstance(cfg, dict):
        for key in ("visualInteractions", "interactions", "relationships"):
            vi2 = cfg.get(key)
            if isinstance(vi2, list):
                interactions += len(vi2)
    # 3) Textual fallback over the whole section JSON
    if interactions == 0:
        sec_blob_str = json.dumps(sec, default=str)
        hits = sec_blob_str.count('"visualInteractions"') + sec_blob_str.count('"interactionType"')
        interactions += hits
    # 4) PBIR per-visual
    for v in pd["visuals"]:
        s = json.dumps(v)
        if '"visualInteractions"' in s or '"interactionType"' in s:
            interactions += 1
L3 = 0
L3 += 7 if n_slicers >= 2 else (4 if n_slicers == 1 else 0)
L3 += 7 if sync_count >= 1 else 0
L3 += 6 if interactions >= 1 else 0
L3 = min(20, L3)
print(f"🟠 L3 Interactivity:   slicers={n_slicers}  sync={sync_count}  interactions={interactions}  →  {L3}/20")

# ---------------- Level 4 ---------------- #
bookmarks_json = _json("definition/bookmarks/bookmarks.json")
n_bookmarks = 0
if isinstance(bookmarks_json, dict):
    n_bookmarks = len(bookmarks_json.get("items", []))
elif isinstance(bookmarks_root, list):
    n_bookmarks = len(bookmarks_root)
elif isinstance(bookmarks_root, dict):
    n_bookmarks = len(bookmarks_root.get("items", []) or bookmarks_root.get("children", []))
else:
    # scan folder (PBIR)
    n_bookmarks = sum(1 for p in parts if p.startswith("definition/bookmarks/") and p.endswith("/bookmark.json"))
n_tooltips      = sum(1 for pd in pages_data if page_kind(pd["json"]) == "tooltip")
n_drillthrough  = sum(1 for pd in pages_data if page_kind(pd["json"]) == "drillthrough")
L4 = 0
L4 += 7 if n_bookmarks >= 1 else 0
L4 += 7 if n_tooltips >= 1 else 0
L4 += 6 if n_drillthrough >= 1 else 0
L4 = min(20, L4)
print(f"🟣 L4 Storytelling:    bookmarks={n_bookmarks}  tooltips={n_tooltips}  drillthrough={n_drillthrough}  →  {L4}/20")

# ---------------- Level 5 ---------------- #
# conditional formatting: look for "objects" with "dataBars" / "background" / "fontColor" with "fillRule"/"gradient"
cond_fmt = 0
for v in all_visuals:
    s = json.dumps(v)
    if any(k in s for k in ('"dataBars"', '"colorScale"', '"fillRule"', '"gradient"')):
        cond_fmt += 1
n_mobile = sum(1 for pd in pages_data if page_has_mobile(pd["json"]))
L5 = 0
L5 += 7 if has_custom_theme else 0
L5 += 7 if cond_fmt >= 1 else 0
L5 += 6 if n_mobile >= 1 else 0
L5 = min(20, L5)
print(f"🔵 L5 Polish:          theme={has_custom_theme}  condFmt={cond_fmt}  mobile={n_mobile}  →  {L5}/20")

TOTAL = L1 + L2 + L3 + L4 + L5
print()
print("=" * 60)
print(f"  TOTAL SCORE: {TOTAL}/100")
print("=" * 60)

if   TOTAL >= 100: RANK = "Kill Screen Survivor"
elif TOTAL >=  80: RANK = "Arcade Legend"
elif TOTAL >=  60: RANK = "High Roller"
elif TOTAL >=  40: RANK = "Quarter Muncher"
elif TOTAL >=  20: RANK = "Newbie"
else:              RANK = "Spectator"

FINAL_SCORE = TOTAL
FINAL_RANK  = RANK
print(f"  RANK: {RANK}")


## Step 5 — 🏅 Mint your shareable badge


In [ ]:
# ============================================================
# Retro Arcade — Badge issuance
# HMAC-signed URL for the GitHub Pages badge viewer
# ============================================================
import json, time, hmac, hashlib, base64
from IPython.display import display, Markdown, HTML

_BADGE_SECRET = b"fabric-arcade-badge-v1-7K9mP3xQ"
_BASE_URL     = "https://maenglar78.github.io/fabric-arcade"
_GAME_ID      = "retro-arcade"
_SKILLS       = ["Power BI", "Direct Lake", "Lakehouse"]

def _b64u(b: bytes) -> str:
    return base64.urlsafe_b64encode(b).rstrip(b"=").decode("ascii")

def _issue(game_id, player, rank, score):
    payload = {"v": 1, "g": game_id, "p": str(player),
               "r": str(rank), "s": int(score), "t": int(time.time()),
               "k": _SKILLS}
    body = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    sig  = hmac.new(_BADGE_SECRET, body, hashlib.sha256).digest()
    return f"{_BASE_URL}/badge.html?t={_b64u(body)}.{_b64u(sig)}"

score = globals().get("FINAL_SCORE", 0)
rank  = globals().get("FINAL_RANK", "Spectator")

if score < 20:
    display(Markdown(
        f"### 🚧 Not yet eligible (score {score}/100)\n\n"
        f"Reach **at least 20 points** to earn the Newbie badge. "
        f"Re-open `02_Quest` for the level checklist."
    ))
elif PLAYER_NAME.strip() in ("", "Your Name Here"):
    display(Markdown(
        "### ✍️ Set your name first\n\n"
        "Edit `PLAYER_NAME` in **Step 0** and re-run the notebook."
    ))
else:
    url = _issue(_GAME_ID, PLAYER_NAME, rank, score)
    display(Markdown(
        f"### 🏅 Badge minted\n\n"
        f"**{PLAYER_NAME}** — *{rank}* · score **{score}/100**\n\n"
        f"🔗 **[Open your badge]({url})**\n\n"
        f"Click *Download PNG* / *Share on LinkedIn* on the badge page."
    ))
    display(HTML(f'<a href="{url}" target="_blank" '
                 f'style="display:inline-block;padding:10px 20px;border-radius:8px;'
                 f'background:linear-gradient(135deg,#ff006e,#8338ec);color:white;'
                 f'text-decoration:none;font-weight:600">🏅 Open my badge page</a>'))
